# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 13

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. Аналитик выполнил две строчки кода:
```python
s1 = df['balance']
s2 = df[['balance']]
```
В какой переменной (`s1` или `s2`) получен одномерный столбец (`pd.Series`), а в какой — двумерная таблица (`pd.DataFrame`) из одной колонки? В чем их главное отличие при обращении к методам?

**Ответ:**
* `s1 = df['balance']` возвращает одномерную структуру `pd.Series` (вектор значений).
* `s2 = df[['balance']]` (список в двойных квадратных скобках) возвращает двумерную таблицу `pd.DataFrame` из одной колонки.
К `Series` обращаются как к одномерному вектору, а к `DataFrame` применимы табличные методы (выборка нескольких колонок, отображение в виде таблицы).

---

#### 1.2. Аналитик вызвал `df.info()` для таблицы игровых сессий из 500 строк и увидел запись:
`client_rating: 425 non-null float64`.
Сколько сессий в таблице остались без клиентской оценки (содержат `NaN` / `None`), и какой метод Pandas позволяет подсчитать количество пропусков во всех столбцах?

**Ответ:**
Без оценки остались **75 сессий** (500 - 425 = 75). Для подсчета точного количества пропусков в каждом столбце используется метод `df.isna().sum()` (или `df.isnull().sum()`).

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["prize_multiplier"] = (df["prize_fund"] / df["entry_fee"]).round(1)
```

**Ответ:**
Рассчитывает коэффициент отдачи призового фонда турнира по отношению к вступительному взносу (во сколько раз призовой фонд превышает билет).

---

#### 1.4. Аналитик выполнил расчет средней суммы оплат по играм:
```python
report = df.groupby('game_title')['total_paid'].mean()
```
Сколько строк получится в объекте `report` (столько же, сколько всех сессий, или столько, сколько уникальных игр)? Что стало индексом строк в полученном объекте?

**Ответ:**
В объекте `report` строк будет ровно столько, сколько **уникальных игр** в таблице (строки свернулись по уникальным значениям `game_title`). Новым индексом строк стали сами названия игр (`game_title`).

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `game_sessions.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path("../variant_13")
df_audit = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Максимальная длительность сессии: 2.21")

Количество строк в таблице: 500
Столбцы датафрейма: ['session_id', 'member_id', 'pc_id', 'game_title', 'session_date', 'duration_hours', 'discount_promo', 'total_paid', 'client_rating']
Максимальная длительность сессии: 2.21

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_13")
df_data = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

summary = df_data.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

summary["quantity"] = summary["quantity"].round(2)
summary = summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(summary)

,category,total_amount,quantity
0,Мерч,83000,1.44
1,Напитки,31190,1.48
2,Горячее,30970,1.66
3,Снеки,20220,1.68


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `members` поле `member_id` объявлено как `PRIMARY KEY` (первичный ключ).
Что произойдет, если система попытается добавить нового клиента с уже существующим номером `member_id = 1` или со значением `NULL`? Почему СУБД вернет ошибку?

**Ответ:**
СУБД заблокирует операцию и вернет ошибку нарушения ограничения целостности (`PRIMARY KEY / UNIQUE / NOT NULL constraint failed`). Значение первичного ключа обязано быть уникальным для каждой записи и не может быть пустым (`NULL`), так как оно однозначно идентифицирует строку.

---

#### 3.2. Администратор киберклуба случайно выполнил запрос `DELETE FROM game_sessions;` вместо удаления одной устаревшей сессии.
Если запрос выполнялся внутри открытой транзакции, какой командой можно отменить удаление до того, как изменения зафиксируются на диске?

**Ответ:**
Командой `ROLLBACK`. Она отменяет все операции текущей открытой транзакции и возвращает данные в исходное состояние (до вызова команды `COMMIT`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT item_name, SUM(quantity) AS total_sold, SUM(total_amount) AS revenue FROM bar_orders GROUP BY item_name ORDER BY total_sold DESC LIMIT 5;
```

**Ответ:**
Запрос группирует заказы бара по наименованию товара, рассчитывая суммарное количество проданных штук и общую выручку, и выводит топ-5 самых продаваемых товаров.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_13")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE loyalty_tier = 'Bronze' AND balance > 0
ORDER BY balance DESC, member_id ASC
LIMIT 7;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,member_id,nickname,city,loyalty_tier,balance
0,22,CyberKnight_964,Казань,Bronze,7500.0
1,71,Paladin_886,Новосибирск,Bronze,7500.0
2,103,Grizzly_163,Москва,Bronze,7500.0
3,133,Flash_789,Москва,Bronze,7500.0
4,1,Thunder_155,Москва,Bronze,4000.0
5,15,Neo_122,Санкт-Петербург,Bronze,4000.0
6,36,Striker_321,Новосибирск,Bronze,4000.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT game_title, COUNT(*) AS sessions_count, ROUND(AVG(total_paid), 2) AS avg_paid, ROUND(SUM(duration_hours), 2) AS total_hours
FROM game_sessions
GROUP BY game_title
HAVING AVG(total_paid) > 400
ORDER BY total_hours DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,game_title,sessions_count,avg_paid,total_hours
0,Warzone,54,532.76,125.0
1,Dota 2,57,425.98,125.0
2,Cyberpunk 2077,56,514.13,123.5
3,Valorant,55,427.42,119.5
4,Rust,54,463.41,112.0
5,Atomic Heart,44,493.82,109.0
6,Apex Legends,48,498.21,106.5
7,PUBG,41,503.78,101.0
8,Counter-Strike 2,50,421.16,99.5
9,GTA V,41,443.00,85.5


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, ROUND(SUM(s.duration_hours), 2) AS dota_hours, ROUND(SUM(s.total_paid), 2) AS dota_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
WHERE s.game_title = 'Dota 2'
GROUP BY m.member_id, m.nickname
ORDER BY dota_spent DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,dota_hours,dota_spent
0,18,Sniper_313,3.0,1140.0
1,88,Kraken_169,4.0,1140.0
2,118,Ranger_85,6.5,1135.0
3,56,Kaiser_272,7.0,1110.0
4,117,Bullet_396,6.0,1020.0
5,114,Reaper_714,4.0,942.0
6,86,Cobra_878,4.0,900.0
7,63,Blade_594,4.5,882.0
8,98,Storm_899,3.0,802.0
9,43,Lynx_505,4.0,800.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT session_id, member_id, game_title, duration_hours, total_paid
FROM game_sessions
WHERE game_title = 'Counter-Strike 2' AND duration_hours > (SELECT AVG(duration_hours) FROM game_sessions WHERE game_title = 'Counter-Strike 2')
ORDER BY duration_hours DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,session_id,member_id,game_title,duration_hours,total_paid
0,20043,8,Counter-Strike 2,4.0,680.0
1,20044,36,Counter-Strike 2,4.0,1080.0
2,20101,75,Counter-Strike 2,4.0,680.0
3,20123,98,Counter-Strike 2,4.0,600.0
4,20405,47,Counter-Strike 2,4.0,920.0
5,20022,76,Counter-Strike 2,3.0,450.0
6,20160,100,Counter-Strike 2,3.0,810.0
7,20275,105,Counter-Strike 2,3.0,450.0
8,20256,14,Counter-Strike 2,2.5,338.0
9,20032,56,Counter-Strike 2,2.0,760.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Призовой фонд по игровым дисциплинам

**Найденные ошибки:**
1. Пропущена группировка: поле `game_title` выбирается вместе с агрегатными функциями `COUNT` и `SUM`, но предложение `GROUP BY game_title` отсутствует.

In [9]:
query_51_fixed = '''
SELECT 
    game_title,
    COUNT(*) AS tourn_count,
    SUM(prize_fund) AS total_prizes
FROM tournaments
GROUP BY game_title
ORDER BY total_prizes DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,game_title,tourn_count,total_prizes
0,Counter-Strike 2,2,170000.0
1,Dota 2,2,105000.0
2,PUBG,1,45000.0
3,Apex Legends,1,40000.0
4,Valorant,1,35000.0
5,Cyberpunk 2077,1,25000.0


#### 5.2. Призовой фонд турниров по дисциплинам и рангам

**Найденные ошибки:**
1. Агрегатная функция в `WHERE`: фильтрация по `SUM(prize_fund) > 20000` должна выполняться в `HAVING`.
2. Неполный `GROUP BY`: столбец `min_rank` выбран в `SELECT`, но отсутствует в списке группировки.

In [10]:
query_52_fixed = '''
SELECT 
    game_title,
    min_rank,
    SUM(prize_fund) AS total_prizes
FROM tournaments
GROUP BY game_title, min_rank
HAVING SUM(prize_fund) > 20000
ORDER BY total_prizes DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,game_title,min_rank,total_prizes
0,Counter-Strike 2,Gold,120000.0
1,Dota 2,Silver,75000.0
2,Counter-Strike 2,Bronze,50000.0
3,PUBG,Silver,45000.0
4,Apex Legends,Bronze,40000.0
5,Valorant,Bronze,35000.0
6,Dota 2,Bronze,30000.0
7,Cyberpunk 2077,Bronze,25000.0


#### 5.3. Сравнение расходов в баре со средним чеком бара

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = b.order_id` связывает клиента с номером заказа вместо `b.member_id`.
2. Неполный `GROUP BY`: поле `m.nickname` выбрано в `SELECT`, но отсутствует в списке группировки.
3. Многострочный подзапрос: группировка `GROUP BY category` в подзапросе возвращает набор строк вместо одного скалярного среднего.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    ROUND(AVG(b.total_amount), 2) AS avg_order_sum
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
GROUP BY m.member_id, m.nickname
HAVING AVG(b.total_amount) > (
    SELECT AVG(total_amount) 
    FROM bar_orders
)
ORDER BY avg_order_sum DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,avg_order_sum
0,67,Shade_796,2700.0
1,121,Viper_138,2400.0
2,137,Shadow_92,1500.0
3,51,Striker_259,1290.0
4,48,Storm_780,1178.0
5,45,Frost_149,1125.0
6,113,Paladin_50,1058.0
7,133,Flash_789,970.0
8,71,Paladin_886,945.0
9,58,Dragon_918,940.0
